# QLoRA Adapter Training

Train per-artist LoRA/DoRA adapters on Gemma 4 E4B.

In [1]:
from huggingface_hub import login
login()

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="google/gemma-4-E4B",
    local_dir="./models/gemma-4-E4B",
)

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

'/home/aliozkaya/uni/467/term_project/src/models/gemma-4-E4B'

In [ ]:
import pandas as pd
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

from config import PROMPT
from models import load_base_model
from style_loss import build_style_weights, top_tokens, make_style_loss_func

model, tokenizer = load_base_model()
model = prepare_model_for_kbit_training(model)

train_df = pd.read_csv("./data/train.csv")
train_df["text"] = train_df.apply(lambda row: f"{PROMPT}{row['clean']}", axis=1)

In [4]:
def train_adapter(artist, r=8, use_dora=False, epochs=3, lr=2e-4, style_weights=None):
    artist_df = train_df[train_df["artist"] == artist].reset_index(drop=True)
    dataset = Dataset.from_pandas(artist_df[["text"]])

    tag = "dora" if use_dora else "lora"
    suffix = "_sw" if style_weights is not None else ""
    output_dir = f"./adapters/{artist.lower().replace(' ', '_')}_{tag}_r{r}{suffix}"

    lora_config = LoraConfig(
        r=r,
        lora_alpha=r * 2,
        target_modules=r"model\.language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)",
        lora_dropout=0.1,
        bias="none",
        task_type="CAUSAL_LM",
        use_dora=use_dora,
    )

    peft_model = get_peft_model(model, lora_config)

    training_args = SFTConfig(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,
        learning_rate=lr,
        max_length=512,
        bf16=True,
        logging_steps=5,
        save_strategy="epoch",
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        gradient_checkpointing=True,
        report_to="none",
        weight_decay=0.05,
    )

    trainer = SFTTrainer(
        model=peft_model,
        train_dataset=dataset,
        args=training_args,
        processing_class=tokenizer,
        compute_loss_func=make_style_loss_func(style_weights) if style_weights is not None else None,
    )

    trainer.train()
    peft_model.save_pretrained(output_dir)
    peft_model.unload()
    print(f"Saved: {output_dir} ({len(artist_df)} songs, {tag}, r={r}{suffix})")
    return output_dir

## Main adapters (LoRA)

In [5]:
train_adapter("Gojira", r=8, use_dora=False)
train_adapter("Tool", r=8, use_dora=False)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/78 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/78 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 1}.


Step,Training Loss
5,4.159589
10,3.519953
15,3.222866
20,3.084101
25,2.881937
30,2.748808
35,2.933267
40,2.923807
45,2.524714
50,2.463408


...

Adding EOS to train dataset:   0%|          | 0/67 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/67 [00:00<?, ? examples/s]

Step,Training Loss
5,3.023287
10,2.228918
15,1.936412
20,1.883433
25,2.087738
30,1.846724
35,1.693688
40,1.612159
45,1.527674
50,1.245156


Saved: ./adapters/tool_lora_r8 (67 songs, lora, r=8)


'./adapters/tool_lora_r8'

## Ablation: DoRA (same artist)

In [6]:
train_adapter("Gojira", r=8, use_dora=True)
train_adapter("Tool", r=8, use_dora=True)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/78 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/78 [00:00<?, ? examples/s]

Step,Training Loss
5,4.158524
10,3.520868
15,3.223913
20,3.080611
25,2.877343
30,2.725910
35,2.922875
40,2.896684
45,2.507811
50,2.445800


Saved: ./adapters/gojira_dora_r8 (78 songs, dora, r=8)


...

Adding EOS to train dataset:   0%|          | 0/67 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/67 [00:00<?, ? examples/s]

Step,Training Loss
5,3.020968
10,2.221585
15,1.933254
20,1.897352
25,2.087187
30,1.840265
35,1.688256
40,1.606612
45,1.511502
50,1.248870


Saved: ./adapters/tool_dora_r8 (67 songs, dora, r=8)


'./adapters/tool_dora_r8'

## Ablation: Rank

In [7]:
train_adapter("Gojira", r=4, use_dora=False)
train_adapter("Gojira", r=16, use_dora=False)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/78 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/78 [00:00<?, ? examples/s]

Step,Training Loss
5,4.248919
10,3.645770
15,3.275770
20,3.138378
25,2.991206
30,2.850618
35,3.023703
40,3.030144
45,2.706154
50,2.678022


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...

Adding EOS to train dataset:   0%|          | 0/78 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/78 [00:00<?, ? examples/s]

Step,Training Loss
5,4.080651
10,3.444357
15,3.192137
20,3.055118
25,2.783633
30,2.639443
35,2.903032
40,2.779564
45,2.336172
50,2.213894


Saved: ./adapters/gojira_lora_r16 (78 songs, lora, r=16)


'./adapters/gojira_lora_r16'

## Experiment: Style-Weighted Loss

Up-weight artist-distinctive tokens (token-level TF-IDF) in the cross-entropy so
the adapter learns artist vocabulary, not just genre. Building/inspecting the
weights is CPU-only; only the `train_adapter` call below needs the GPU. Trains to
`*_sw` adapters so they can be A/B'd against the plain ones on attribution.

In [ ]:
style_w = build_style_weights("Gojira", train_df, tokenizer, text_col="clean")
top_tokens(style_w, tokenizer, n=30)

In [ ]:
train_adapter("Gojira", r=8, use_dora=False, style_weights=style_w)